In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow
import mlflow.keras
from mlflow.models.signature import infer_signature
import os
import warnings
warnings.filterwarnings('ignore')

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the data set
df = pd.read_csv("../data/filtered_icfes_data_cesar.csv")
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

print(f"Dimensiones: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")

Dimensiones: (63127, 28)

Columnas: ['periodo', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_mcpio_ubicacion', 'cole_jornada', 'cole_naturaleza', 'estu_cod_reside_mcpio', 'estu_genero', 'estu_nacionalidad', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'punt_ingles', 'punt_matematicas', 'punt_sociales_ciudadanas', 'punt_c_naturales', 'punt_lectura_critica', 'punt_global']


In [4]:
df = df.astype({
    "periodo": "float32",
    "cole_area_ubicacion": "category",
    "cole_bilingue": "category",
    "cole_calendario": "category",
    "cole_caracter": "category",
    "cole_cod_dane_establecimiento": "category",
    "cole_cod_dane_sede": "category",
    "cole_cod_mcpio_ubicacion": "category",
    "cole_jornada": "category",
    "cole_naturaleza": "category",
    "estu_cod_reside_mcpio": "category",
    "estu_genero": "category",
    "estu_nacionalidad": "category",
    "fami_cuartoshogar": "category",
    "fami_educacionmadre": "category",
    "fami_educacionpadre": "category",
    "fami_estratovivienda": "category",
    "fami_personashogar": "category",
    "fami_tieneautomovil": "category",
    "fami_tienecomputador": "category",
    "fami_tieneinternet": "category",
    "fami_tienelavadora": "category",
})

# Binarias a bool
binary_map = {
    "cole_area_ubicacion":  {"URBANO": True,  "RURAL": False},
    "cole_bilingue":        {"S": True,        "N": False},
    "cole_calendario":      {"A": True,        "B": False},
    "cole_naturaleza":      {"OFICIAL": True,  "NO OFICIAL": False},
    "estu_genero":          {"M": True,        "F": False},
    "fami_tieneautomovil":  {"Si": True,       "No": False},
    "fami_tienecomputador": {"Si": True,       "No": False},
    "fami_tieneinternet":   {"Si": True,       "No": False},
    "fami_tienelavadora":   {"Si": True,       "No": False},
}
for col, mapping in binary_map.items():
    df[col] = df[col].map(mapping)

df = df.rename(columns={
    "cole_area_ubicacion": "cole_area_urbano",
    "cole_calendario":     "cole_calendario_a",
    "cole_naturaleza":     "cole_oficial",
    "estu_genero":         "estu_masculino",
})

cuartos_map = {
    "Uno": "1", "Dos": "2", "Tres": "3", "Cuatro": "4", "Cinco": "5",
    "Seis": "6+", "Seis o mas": "6+", "Siete": "6+",
    "Ocho": "6+", "Nueve": "6+", "Diez o más": "6+"
}
df["fami_cuartoshogar"] = df["fami_cuartoshogar"].map(cuartos_map)

personas_map = {
    "Una": "1 a 2", "Dos": "1 a 2",
    "Tres": "3 a 4", "Cuatro": "3 a 4",
    "Cinco": "5 a 6", "Seis": "5 a 6",
    "Siete": "7 a 8", "Ocho": "7 a 8",
    "Nueve": "9 o más", "Diez": "9 o más",
    "Once": "9 o más", "Doce o más": "9 o más"
}
df["fami_personashogar"] = df["fami_personashogar"].map(personas_map)

categorical_cols = [
    "cole_caracter", "cole_cod_dane_establecimiento", "cole_cod_dane_sede",
    "cole_cod_mcpio_ubicacion", "cole_jornada", "estu_cod_reside_mcpio",
    "estu_nacionalidad", "fami_cuartoshogar", "fami_educacionmadre",
    "fami_educacionpadre", "fami_estratovivienda", "fami_personashogar",
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Dimensiones tras preprocesamiento: {df.shape}")
print(f"Valores nulos restantes: {df.isnull().sum().sum()}")

Dimensiones tras preprocesamiento: (63127, 461)
Valores nulos restantes: 0


In [9]:
TARGET_COLS = ['punt_matematicas', 'punt_lectura_critica',
               'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles']
DROP_COLS   = ['punt_global'] + TARGET_COLS

X = df.drop(columns=DROP_COLS).to_numpy(dtype=np.float32)
y = df[TARGET_COLS].to_numpy(dtype=np.float32)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

X shape: (63127, 455)
y shape: (63127, 5)

X_train: (50501, 455) | X_test: (12626, 455)
y_train: (50501, 5) | y_test: (12626, 5)


In [10]:
tracking_dir = os.path.join(os.getcwd(), "mlruns")
mlflow.set_tracking_uri(f"file:///{tracking_dir.replace(os.sep, '/')}")

EXPERIMENT_NAME = "pregunta3_scores"
mlflow.set_experiment(EXPERIMENT_NAME)

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment ID       : {exp.experiment_id}")
print(f"X_train shape       : {X_train.shape}")
print(f"X_test  shape       : {X_test.shape}")
print(f"Targets             : {TARGET_COLS}")

2026/05/24 13:43:06 INFO mlflow.tracking.fluent: Experiment with name 'pregunta3_scores' does not exist. Creating a new experiment.


MLflow tracking URI : file:///c:/Users/Asus/Desktop/Clases/Analítica/Proy2/IIND4130-P2/data_science_3/mlruns
Experiment ID       : 320118271050873575
X_train shape       : (50501, 455)
X_test  shape       : (12626, 455)
Targets             : ['punt_matematicas', 'punt_lectura_critica', 'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles']
